In [2]:

def edit_distance(s1, s2):
    len_s1 = len(s1)
    len_s2 = len(s2)
    
    # DPテーブルを初期化
    dp = [[0] * (len_s2 + 1) for _ in range(len_s1 + 1)]
    
    # 初期状態を設定
    for i in range(len_s1 + 1):
        dp[i][0] = i  # 変換するために削除する必要がある文字数
    for j in range(len_s2 + 1):
        dp[0][j] = j  # 変換するために追加する必要がある文字数

    # DPテーブルを埋める
    for i in range(1, len_s1 + 1):
        for j in range(1, len_s2 + 1):
            if s1[i - 1] == s2[j - 1]:  # 文字が一致する場合
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(dp[i - 1][j] + 1,    # 削除
                               dp[i][j - 1] + 1,    # 挿入
                               dp[i - 1][j - 1] + 1  # 置換
                              )
    
    logging.info(f'Edit distance calculated for strings: "{s1}" and "{s2}" is {dp[len_s1][len_s2]}')
    return dp[len_s1][len_s2]

def cyclic_edit_distance(seq1, seq2):
    # 円順列のすべての回転を生成
    n = len(seq1)
    rotations = [seq1[i:] + seq1[:i] for i in range(n)]

    # 各回転について編集距離を計算し、最小距離を探索
    min_distance = float('inf')
    for rotation in rotations:
        distance = edit_distance(rotation, seq2)
        if distance < min_distance:
            min_distance = distance

    logging.info(f'Minimum cyclic edit distance for sequences: "{seq1}" and "{seq2}" is {min_distance}')
    return min_distance

def calculate_circular_fit(cycle1, cycle2, isLog=False):
    return cyclic_edit_distance(cycle1, cycle2)


def calculate_circular_fit_with_arrange(cycle1, cycle2, isLog=False):
    distance = calculate_circular_fit(cycle1, cycle2, isLog)
    reverse_distance = calculate_circular_fit(cycle1, reverse_order(cycle2), isLog)
    if (distance > reverse_distance):
        distance = reverse_distance
        cycle2 = reverse_order(cycle2)
    return distance, cycle2

def calculate_circular_fit_with_arranges(cycle1, petalCount, isLog=False):
    min_distance = float('inf')  # Change to min_distance
    fit_arrange = []
    fit_arrange_key = ''
    for key, arrange_list in arranges.items():
        if petalCount != len(arrange_list):
            continue
        distance, arrange = calculate_circular_fit_with_arrange(cycle1, arrange_list, isLog)
        if (min_distance > distance):  # Change to min_distance
            min_distance = distance
            fit_arrange = arrange
            fit_arrange_key = key
    
    return min_distance, fit_arrange, fit_arrange_key  # Change to min_distance


In [ ]:
import numpy as np
import os
from utils import probas_to_scores_and_classes, nms, sort_by_bboxes
from PIL import Image
import logging

# Set up logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def edit_distance(s1, s2):
    len_s1 = len(s1)
    len_s2 = len(s2)
    
    # DPテーブルを初期化
    dp = [[0] * (len_s2 + 1) for _ in range(len_s1 + 1)]
    
    # 初期状態を設定
    for i in range(len_s1 + 1):
        dp[i][0] = i  # 変換するために削除する必要がある文字数
    for j in range(len_s2 + 1):
        dp[0][j] = j  # 変換するために追加する必要がある文字数

    # DPテーブルを埋める
    for i in range(1, len_s1 + 1):
        for j in range(1, len_s2 + 1):
            if s1[i - 1] == s2[j - 1]:  # 文字が一致する場合
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(dp[i - 1][j] + 1,    # 削除
                               dp[i][j - 1] + 1,    # 挿入
                               dp[i - 1][j - 1] + 1  # 置換
                              )
    
    logging.info(f'Edit distance calculated for strings: "{s1}" and "{s2}" is {dp[len_s1][len_s2]}')
    return dp[len_s1][len_s2]

def cyclic_edit_distance(seq1, seq2):
    # 円順列のすべての回転を生成
    n = len(seq1)
    rotations = [seq1[i:] + seq1[:i] for i in range(n)]

    # 各回転について編集距離を計算し、最小距離を探索
    min_distance = float('inf')
    for rotation in rotations:
        distance = edit_distance(rotation, seq2)
        if distance < min_distance:
            min_distance = distance

    logging.info(f'Minimum cyclic edit distance for sequences: "{seq1}" and "{seq2}" is {min_distance}')
    return min_distance

def calculate_circular_fit(cycle1, cycle2, isLog=False):
    return cyclic_edit_distance(cycle1, cycle2)


def calculate_circular_fit_with_arrange(cycle1, cycle2, isLog=False):
    distance = calculate_circular_fit(cycle1, cycle2, isLog)
    reverse_distance = calculate_circular_fit(cycle1, reverse_order(cycle2), isLog)
    if (distance > reverse_distance):
        distance = reverse_distance
        cycle2 = reverse_order(cycle2)
    return distance, cycle2

def calculate_circular_fit_with_arranges(cycle1, petalCount, isLog=False):
    min_distance = float('inf')  # Change to min_distance
    fit_arrange = []
    fit_arrange_key = ''
    for key, arrange_list in arranges.items():
        if petalCount != len(arrange_list):
            continue
        distance, arrange = calculate_circular_fit_with_arrange(cycle1, arrange_list, isLog)
        if (min_distance > distance):  # Change to min_distance
            min_distance = distance
            fit_arrange = arrange
            fit_arrange_key = key
    
    return min_distance, fit_arrange, fit_arrange_key  # Change to min_distance


def init_counters():
    """カウンターの初期化"""
    keys = list(arranges.keys()) + ['others']
    return {
        'arrange_key_counts': {key: 0 for key in keys},
        'arrange_key_counts_ground_truth': {key: 0 for key in keys},
        'key_match_counts': {key: 0 for key in keys},
        'key_mismatch_counts': {key: 0 for key in keys},
        'others_patterns': [],
        'others_patterns_ground_truth': []
    }

def update_counters(counters, min_fit, fit_arrange_key, classes, is_ground_truth=False):  # Change max_fit to min_fit
    """カウンターの更新"""
    if min_fit < 0:  # Change to min_fit
        if is_ground_truth:
            counters['arrange_key_counts_ground_truth'][fit_arrange_key] += 1
        else:
            counters['arrange_key_counts'][fit_arrange_key] += 1
    else:
        if is_ground_truth:
            counters['arrange_key_counts_ground_truth']['others'] += 1
            counters['others_patterns_ground_truth'].append(classes)
        else:
            counters['arrange_key_counts']['others'] += 1
            counters['others_patterns'].append(classes)

def update_match_counters(counters, pred_fit, true_fit, pred_key, true_key):
    """マッチングカウンターの更新"""
    if pred_fit < 0 and true_fit < 0:  # Both predictions are negative
        if pred_key == true_key:
            counters['key_match_counts'][pred_key] += 1
        else:
            counters['key_mismatch_counts'][pred_key] += 1
            counters['key_mismatch_counts'][true_key] += 1
    elif pred_fit == 0 and true_fit == 0:  # Both predictions are zero
        counters['key_match_counts']['others'] += 1
    elif pred_fit < 0:  # Only predicted fit is negative
        counters['key_mismatch_counts']['others'] += 1
        counters['key_mismatch_counts'][true_key] += 1
    elif true_fit < 0:  # Only true fit is negative
        counters['key_mismatch_counts']['others'] += 1
        counters['key_mismatch_counts'][pred_key] += 1
    else:  # Both fits are positive
        counters['key_mismatch_counts']['others'] += 1
        counters['key_mismatch_counts'][pred_key] += 1
        counters['key_mismatch_counts'][true_key] += 1
        if pred_key == true_key:  # Increment match count if keys are the same
            counters['key_match_counts'][pred_key] += 1

def plot_results_graphs(counters, confusion_matrix):
    """結果のグラフ描画"""
    keys = sorted(list(arranges.keys()) + ['others'], reverse=True)
    plt_settings = {
        'figsize': (16, 10),
        'xlabel': 'Frequency',
        'ylabel': 'Arrangement Pattern'
    }

    # 予測結果のグラフ
    plt.figure(figsize=plt_settings['figsize'])
    plt.barh(keys, [counters['arrange_key_counts'][key] for key in keys])
    plt.title('Predicted Arrangement Pattern Distribution')
    plt.xlabel(plt_settings['xlabel'])
    plt.ylabel(plt_settings['ylabel'])
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

    # 正解データのグラフ
    plt.figure(figsize=plt_settings['figsize'])
    plt.barh(keys, [counters['arrange_key_counts_ground_truth'][key] for key in keys])
    plt.title('Ground Truth Arrangement Pattern Distribution')
    plt.xlabel(plt_settings['xlabel'])
    plt.ylabel(plt_settings['ylabel'])
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

    # Match/Mismatchグラフ
    plt.figure(figsize=plt_settings['figsize'])
    x = np.arange(len(keys))
    width = 0.35

    accuracies = {
        key: (counters['key_match_counts'][key] / (counters['arrange_key_counts_ground_truth'][key]) * 100 
              if counters['arrange_key_counts_ground_truth'][key] > 0 else 0)
        for key in keys
    }

    plt.barh(x - width/2, [counters['key_match_counts'][key] for key in keys], width, label='Match')
    plt.barh(x + width/2, [counters['key_mismatch_counts'][key] for key in keys], width, label='Mismatch')

    for i, key in enumerate(keys):
        plt.text(max(counters['key_match_counts'][key], counters['key_mismatch_counts'][key]) + 1, i, 
                f'{accuracies[key]:.1f}%', va='center')

    plt.yticks(x, keys)
    plt.title('Predicted and Ground Truth Key Match/Mismatch Counts')
    plt.xlabel(plt_settings['xlabel'])
    plt.ylabel(plt_settings['ylabel'])
    plt.legend()
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

    # 混同行列のヒートマップ
    heatmap_keys = sorted(list(arranges.keys()) + ['others'])
    plt.figure(figsize=(16, 10))
    plt.imshow(confusion_matrix, cmap='YlOrRd')
    plt.colorbar()

    plt.xticks(np.arange(len(heatmap_keys)), heatmap_keys, rotation=45, ha='right')
    plt.yticks(np.arange(len(heatmap_keys)), heatmap_keys)

    plt.title('Confusion Matrix of Arrangement Patterns')
    plt.xlabel('Ground Truth')
    plt.ylabel('Predicted')

    for i in range(len(heatmap_keys)):
        for j in range(len(heatmap_keys)):
            plt.text(j, i, int(confusion_matrix[i, j]), ha="center", va="center", color="black")

    plt.tight_layout()
    plt.show()


# メイン処理
ENABLE_PRINT = False
ENABLE_PLOT = False

# カウンターの初期化
counters = init_counters()
heatmap_keys = sorted(list(arranges.keys()) + ['others'])
key_to_idx = {key: i for i, key in enumerate(heatmap_keys)}
confusion_matrix = np.zeros((len(heatmap_keys), len(heatmap_keys)))

total_stats = {
    'total_samples': 0,
    'correct_predictions': 0,
    'total_fit_score': 0,
    'total_fit_score_ground_truth': 0
}

all_classes = []
all_sorted_classes = []
all_scores = []

filtered_results = results

# 結果の処理
for filename, classes, scores, bboxes in filtered_results:
    total_stats['total_samples'] += 1
    

    # 正解データの処理
    annotation_classes, annotation_bboxes = get_ground_truth(filename)
    sorted_bboxes, _, sorted_classes = sort_by_bboxes(annotation_bboxes, [0]*len(annotation_bboxes), annotation_classes)
    all_sorted_classes.append(sorted_classes)
    all_scores.append(scores)

    min_fit_known_ground_truth, fit_arrange_known_ground_truth, fit_arrange_key_known_ground_truth = calculate_circular_fit_with_arranges(sorted_classes, len(sorted_classes))
    total_stats['total_fit_score_ground_truth'] += min_fit_known_ground_truth
    update_counters(counters, min_fit_known_ground_truth, fit_arrange_key_known_ground_truth, sorted_classes, True)
    
    # 正解データのカウンターを増やす
    counters['arrange_key_counts_ground_truth'][fit_arrange_key_known_ground_truth] += 1

    # 予測結果の処理
    min_fit_known, fit_arrange_known, fit_arrange_key_known = calculate_circular_fit_with_arranges(classes, len(sorted_classes))
    total_stats['total_fit_score'] += min_fit_known
    all_classes.append(classes)
    update_counters(counters, min_fit_known, fit_arrange_key_known, classes)

    # 混同行列の更新
    pred_key = fit_arrange_key_known
    true_key = fit_arrange_key_known_ground_truth
    confusion_matrix[key_to_idx[pred_key]][key_to_idx[true_key]] += 1

    # 正解判定
    if pred_key == true_key:
        total_stats['correct_predictions'] += 1

    # マッチングカウンターの更新
    update_match_counters(counters, min_fit_known, min_fit_known_ground_truth, 
                         fit_arrange_key_known, fit_arrange_key_known_ground_truth)

    if ENABLE_PLOT:
        img = Image.open(filename)
        img = img.resize((256, 256))  # 前処理のサイズに変形
        plot_results(img, classes, scores, bboxes)

# 精度評価指標の計算と出力
print("\n=== 精度評価指標 ===")
print(f"総サンプル数: {total_stats['total_samples']}")
print(f"正解数: {total_stats['correct_predictions']}")
print(f"正解率: {(total_stats['correct_predictions'] / total_stats['total_samples']) * 100:.2f}%")
print(f"平均適合度スコア(予測): {total_stats['total_fit_score'] / total_stats['total_samples']:.2f}")
print(f"平均適合度スコア(正解): {total_stats['total_fit_score_ground_truth'] / total_stats['total_samples']:.2f}")

# パターンごとの精度
print("\n=== パターンごとの精度 ===")
for key in arranges.keys():
    total = counters['arrange_key_counts_ground_truth'][key]  # Use ground truth count for accuracy
    if total > 0:
        accuracy = (counters['key_match_counts'][key] / total) * 100
        print(f"{key}: {accuracy:.2f}% ({counters['key_match_counts'][key]}/{total})")
    else:
        print(f"{key}: データなし")

# 混同行列の表示
print("\n=== 混同行列 ===")
print("予測＼正解", end="\t")
for key in heatmap_keys:
    print(f"{key[:4]}", end="\t")
print()
for i, pred_key in enumerate(heatmap_keys):
    print(f"{pred_key[:4]}", end="\t")
    for j in range(len(heatmap_keys)):
        print(f"{int(confusion_matrix[i][j])}", end="\t")
    print()

# othersのパターンを表示
print("\n=== othersのパターン(上位10個) ===")
print("\n予測結果のothersパターン:")
for pattern in counters['others_patterns'][:10]:
    print(pattern)

print("\n正解データのothersパターン:")
for pattern in counters['others_patterns_ground_truth'][:10]:
    print(pattern)

# グラフの描画
plot_results_graphs(counters, confusion_matrix)

In [9]:
import logging

arranges = {
    'a1': [2, 0, 2, 0],
    'a2': [1, 2, 1, 0],
    'a3': [1, 1, 2, 0],
    'b1': [1, 0, 2, 0, 2],
    'c1': [1, 0, 1, 2, 0, 2],
    'c2': [0, 2, 0, 2, 0, 2],
    'c3': [2, 1, 0, 2, 0, 1],
    'd1': [2, 1, 0, 2, 1, 0, 1],
    'd2': [0, 2, 0, 1, 2, 0, 2],
    'e1': [2, 1, 0, 1, 2, 0, 2, 0],
    'e2': [0, 2, 0, 2, 0, 2, 0, 2],
    'e3': [0, 2, 0, 2, 1, 0, 2, 1],
    'f1': [0, 2, 0, 2, 0, 2, 0, 1, 2],
    'g1': [2, 0, 2, 0, 2, 0, 2, 0, 2, 0]
}

def reverse_order(arr):
    return arr[::-1]


def calculate_circular_fit_with_arranges(cycle1, petalCount, isLog=False):
    min_distance = float('inf')  # Change to min_distance
    fit_arrange = []
    fit_arrange_key = ''
    for key, arrange_list in arranges.items():
        if petalCount != len(arrange_list):
            continue
        distance, arrange = calculate_circular_fit_with_arrange(cycle1, arrange_list, isLog)
        if (min_distance > distance):  # Change to min_distance
            min_distance = distance
            fit_arrange = arrange
            fit_arrange_key = key
    
    return min_distance, fit_arrange, fit_arrange_key  # Change to min_distance

# Test calculate_circular_fit_with_arranges
for key, arrange in arranges.items():
    petal_count = len(arrange)
    distance, fit_arrange, fit_arrange_key = calculate_circular_fit_with_arranges(arrange, len(arrange))
    print(f"Key: {key}, Distance: {distance}, Fit Arrange: {fit_arrange}, Fit Arrange Key: {fit_arrange_key}")


Key: a1, Distance: 0, Fit Arrange: [2, 0, 2, 0], Fit Arrange Key: a1
Key: a2, Distance: 0, Fit Arrange: [1, 2, 1, 0], Fit Arrange Key: a2
Key: a3, Distance: 0, Fit Arrange: [1, 1, 2, 0], Fit Arrange Key: a3
Key: b1, Distance: 0, Fit Arrange: [1, 0, 2, 0, 2], Fit Arrange Key: b1
Key: c1, Distance: 0, Fit Arrange: [1, 0, 1, 2, 0, 2], Fit Arrange Key: c1
Key: c2, Distance: 0, Fit Arrange: [0, 2, 0, 2, 0, 2], Fit Arrange Key: c2
Key: c3, Distance: 0, Fit Arrange: [2, 1, 0, 2, 0, 1], Fit Arrange Key: c3
Key: d1, Distance: 0, Fit Arrange: [2, 1, 0, 2, 1, 0, 1], Fit Arrange Key: d1
Key: d2, Distance: 0, Fit Arrange: [0, 2, 0, 1, 2, 0, 2], Fit Arrange Key: d2
Key: e1, Distance: 0, Fit Arrange: [2, 1, 0, 1, 2, 0, 2, 0], Fit Arrange Key: e1
Key: e2, Distance: 0, Fit Arrange: [0, 2, 0, 2, 0, 2, 0, 2], Fit Arrange Key: e2
Key: e3, Distance: 0, Fit Arrange: [0, 2, 0, 2, 1, 0, 2, 1], Fit Arrange Key: e3
Key: f1, Distance: 0, Fit Arrange: [0, 2, 0, 2, 0, 2, 0, 1, 2], Fit Arrange Key: f1
Key: g1, Dist